### The Smart Supplier: Optimizing Orders in a Fluctuating Market

Develop a reinforcement learning agent using dynamic programming to help a Smart Supplier decide which products to manufacture and sell each day to maximize profit. The agent must learn the optimal policy for choosing daily production quantities, considering its limited raw materials and the unpredictable daily demand and selling prices for different products.

#### **Scenario**
 A small Smart Supplier manufactures two simple products: Product A and Product B. Each day, the supplier has a limited amount of raw material. The challenge is that the market demand and selling price for Product A and Product B change randomly each day, making some products more profitable than others at different times. The supplier needs to decide how much of each product to produce to maximize profit while managing their limited raw material.

#### **Objective**
The Smart Supplier's agent must learn the optimal policy π∗ using dynamic programming (Value Iteration or Policy Iteration) to decide how many units of Product A and Product B to produce each day to maximize the total profit over the fixed number of days, given the daily changing market conditions and limited raw material.

### --- 1. Custom Environment Creation (SmartSupplierEnv) ---

In [ ]:
import numpy as np
import random

class SmartSupplierEnv:
    def __init__(self, num_days=10, initial_raw_material=50):
        self.num_days = num_days
        self.initial_raw_material = initial_raw_material

        # Defining market states and their product prices
        # Structure: {Market_State_ID: {'A_price': X, 'B_price': Y}}
        self.market_states = {
            1: {'A_price': 10, 'B_price': 12},  # State 1: Product B more profitable
            2: {'A_price': 15, 'B_price': 8}   # State 2: Product A more profitable
        }

        # Define product raw material costs
        self.product_costs = {'A': 3, 'B': 2}

        # Define actions: (num_A, num_B, raw_material_cost_precalculated)
        # Action ID mapping:
        # 0: Produce_2A_0B
        # 1: Produce_1A_2B
        # 2: Produce_0A_5B
        # 3: Produce_3A_0B
        # 4: Do_Nothing
        self.actions = {
            0: (2, 0, 2 * self.product_costs['A']),
            1: (1, 2, 1 * self.product_costs['A'] + 2 * self.product_costs['B']),
            2: (0, 5, 5 * self.product_costs['B']),
            3: (3, 0, 3 * self.product_costs['A']),
            4: (0, 0, 0)
        }

        # Define state space dimensions
        # Current Day: 1 to num_days
        # Current Raw Material: 0 to initial_raw_material
        # Current Market State: 1 or 2
        self.state_space = (
            range(1, self.num_days + 1),
            range(0, self.initial_raw_material + 1),
            list(self.market_states.keys())
        )

        self.action_space = range(len(self.actions))

        self.current_day = None
        self.current_raw_material = None
        self.current_market_state = None

    def reset(self):
        self.current_day = 1
        self.current_raw_material = self.initial_raw_material
        self.current_market_state = random.choice(list(self.market_states.keys()))
        return (self.current_day, self.current_raw_material, self.current_market_state)

    def step(self, action_id):
        if self.current_day is None:
            raise RuntimeError("Environment has not been reset. Call reset() first.")

        num_a, num_b, raw_material_cost = self.actions[action_id]


        reward = self._calculate_reward(self.current_market_state, action_id)

        # Update raw material if action is feasible
        if self.current_raw_material >= raw_material_cost:
            self.current_raw_material -= raw_material_cost
        else:
            reward = -100

        # Transition to next day
        self.current_day += 1

        # Determining next market state randomly
        next_market_state = random.choice(list(self.market_states.keys()))
        self.current_market_state = next_market_state


        # Determining if episode is finished
        done = self.current_day > self.num_days

        next_state = (self.current_day, self.current_raw_material, self.current_market_state)

        return next_state, reward, done, {}

    # get reward function
    def _calculate_reward(self, market_state_id, action_id):
        market_prices = self.market_states[market_state_id]
        num_a, num_b, raw_material_cost = self.actions[action_id]

        profit_a = num_a * market_prices['A_price']
        profit_b = num_b * market_prices['B_price']

        total_profit = profit_a + profit_b

        # Only subtract cost if action is feasible based on current raw material
        if self.current_raw_material >= raw_material_cost:
             reward = total_profit - raw_material_cost
        else:
            reward = -100

        return reward

### --- 2. Dynamic Programming Implementation (Value Iteration or Policy Iteration) ---

In [ ]:
# Value Iteration function
def value_iteration(env, gamma=0.99, theta=1e-6):
    V = {}
    for day in env.state_space[0]:
        for material in env.state_space[1]:
            for market in env.state_space[2]:
                V[(day, material, market)] = 0.0

    while True:
        delta = 0
        for day in env.state_space[0]:
            for material in env.state_space[1]:
                for market in env.state_space[2]:
                    state = (day, material, market)
                    v = V[state]
                    max_q = -float('inf')

                    # Iterating over all possible actions
                    for action_id in env.action_space:
                        num_a, num_b, raw_material_cost = env.actions[action_id]


                        if material >= raw_material_cost:
                            # Calculating immediate reward
                            market_prices = env.market_states[market]
                            reward = (num_a * market_prices['A_price'] + num_b * market_prices['B_price']) - raw_material_cost

                            # Calculating expected value of the next state
                            # Assuming market state transitions are random (50/50 chance for each market state)
                            next_day = day + 1
                            if next_day > env.num_days:
                                next_state_value = 0
                            else:
                                next_state_value_option1 = V[(next_day, material - raw_material_cost, 1)] if (next_day, material - raw_material_cost, 1) in V else 0
                                next_state_value_option2 = V[(next_day, material - raw_material_cost, 2)] if (next_day, material - raw_material_cost, 2) in V else 0
                                next_state_value = 0.5 * next_state_value_option1 + 0.5 * next_state_value_option2

                            # Calculating Q-value for the current state-action pair
                            q_value = reward + gamma * next_state_value
                        else:
                            q_value = -100

                        # Updating max Q-value for the current state
                        max_q = max(max_q, q_value)

                    # Updating the value function for the current state
                    V[state] = max_q
                    delta = max(delta, abs(v - V[state]))

        # Checking for convergence
        if delta < theta:
            break

    # Deriving the optimal policy from the converged value function
    optimal_policy = {}
    for day in env.state_space[0]:
        for material in env.state_space[1]:
            for market in env.state_space[2]:
                state = (day, material, market)
                best_action = None
                max_q = -float('inf')

                for action_id in env.action_space:
                     num_a, num_b, raw_material_cost = env.actions[action_id]

                     if material >= raw_material_cost:
                        market_prices = env.market_states[market]
                        reward = (num_a * market_prices['A_price'] + num_b * market_prices['B_price']) - raw_material_cost

                        next_day = day + 1
                        if next_day > env.num_days:
                            next_state_value = 0
                        else:
                            next_state_value_option1 = V[(next_day, material - raw_material_cost, 1)] if (next_day, material - raw_material_cost, 1) in V else 0
                            next_state_value_option2 = V[(next_day, material - raw_material_cost, 2)] if (next_day, material - raw_material_cost, 2) in V else 0
                            next_state_value = 0.5 * next_state_value_option1 + 0.5 * next_state_value_option2

                        q_value = reward + gamma * next_state_value
                     else:
                         q_value = -100


                     if q_value > max_q:
                         max_q = q_value
                         best_action = action_id

                optimal_policy[state] = best_action

    return V, optimal_policy

### --- 3. Simulation and Policy Analysis ---

In [ ]:
# simulate policy function - Simulates the learned policy over multiple runs to evaluate performance

def simulate_policy(env, policy, num_simulations=1000):
    total_rewards = []
    for _ in range(num_simulations):
        state = env.reset()
        done = False
        episode_reward = 0
        while not done:
            if state in policy:
                 action = policy[state]
            else:
                 action = 4 # Do Nothing
                 if state[0] > env.num_days:
                    break

            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            state = next_state
        total_rewards.append(episode_reward)
    return np.mean(total_rewards)


# analyze policy function - Analyzes and prints snippets of the learned optimal policy

def analyze_policy(policy, env):
    print(" Policy Analysis Snippets :")

    sample_states = [
        (1, env.initial_raw_material, 1),
        (1, env.initial_raw_material, 2),
        (env.num_days // 2, env.initial_raw_material // 2, 1),
        (env.num_days // 2, env.initial_raw_material // 2, 2),
        (env.num_days, env.initial_raw_material // 4, 1),
        (env.num_days, env.initial_raw_material // 4, 2),
        (env.num_days, 0, 1),
        (env.num_days, 0, 2),

    ]

    action_mapping = {v: k for k, v in env.actions.items()}
    for state in sample_states:
        if state in policy:
            optimal_action_id = policy[state]
            optimal_action_details = env.actions[optimal_action_id]
            action_name = None
            for name, details in env.actions.items():
                if details == optimal_action_details:
                    action_name = list(env.actions.keys())[list(env.actions.values()).index(optimal_action_details)]
                    action_name = {0: 'Produce_2A_0B', 1: 'Produce_1A_2B', 2: 'Produce_0A_5B', 3: 'Produce_3A_0B', 4: 'Do_Nothing'}[optimal_action_id]
                    break

            print(f"State: Day {state[0]}, Material {state[1]}, Market {state[2]} -> Optimal Action: {action_name} {optimal_action_details[:2]}")
        else:
            print(f"State: {state} -> Policy not found")


### --- 4. Impact of Dynamics Analysis ---

In [ ]:
# Discusses the impact of dynamic market prices on the optimal policy.
# This section should primarily be a written explanation in the report.

### Impact of Dynamic Market Prices Analysis

* In Market State 1 (A=10, B=12), the policy favors producing Product B due to its higher price and lower raw material cost.

* In Market State 2 (A=15, B=8), the focus shifts to Product A as it becomes more profitable.

* The policy adapts daily, choosing the product with the best profit-to-resource ratio.

* Resource levels and remaining days also influence decisions, but price differences are the main driver early on.

* Dynamic prices make flexibility essential—static plans miss out on price-based opportunities.

* Dynamic programming helps compute adaptive strategies that maximize long-term profit by responding to market shifts.

In [ ]:
# --- Main Execution ---
env = SmartSupplierEnv()
V, optimal_policy = value_iteration(env)

# Simulate the learned policy
average_reward = simulate_policy(env, optimal_policy, num_simulations=100)
print(f"\n\nAverage total reward over 100 simulations: {average_reward:.2f}")
print("------------------------------------------------------")
# Analyze the learned policy
analyze_policy(optimal_policy, env)



Average total reward over 100 simulations: 241.30
------------------------------------------------------
 Policy Analysis Snippets :
State: Day 1, Material 50, Market 1 -> Optimal Action: Produce_0A_5B (0, 5)
State: Day 1, Material 50, Market 2 -> Optimal Action: Do_Nothing (0, 0)
State: Day 5, Material 25, Market 1 -> Optimal Action: Produce_0A_5B (0, 5)
State: Day 5, Material 25, Market 2 -> Optimal Action: Produce_3A_0B (3, 0)
State: Day 10, Material 12, Market 1 -> Optimal Action: Produce_0A_5B (0, 5)
State: Day 10, Material 12, Market 2 -> Optimal Action: Produce_3A_0B (3, 0)
State: Day 10, Material 0, Market 1 -> Optimal Action: Do_Nothing (0, 0)
State: Day 10, Material 0, Market 2 -> Optimal Action: Do_Nothing (0, 0)
